# Generación de base sintética de mora — Banco ProCredit Ecuador

Notebook reproducible y comentado línea por línea para crear **21.500 registros** con 34 variables, valores faltantes controlados y coherencia entre mora, días de atraso, historial crediticio, capacidad de pago y clasificación de riesgo.

## Objetivo y supuestos

- Los registros son **sintéticos** y no identifican clientes reales.
- `ID` contiene cédulas de 10 dígitos válidas estructuralmente: código provincial, tercer dígito de persona natural y verificador módulo 10.
- `zona` utiliza Quito, Guayaquil, Ambato, Ibarra, Cuenca y Santo Domingo, con ponderaciones publicadas por Banco ProCredit.
- `vivienda` solo admite `PROPIA` o `ARRENDADA`.
- `variacion_depositos_ingresos` siempre es positiva cuando está observada.
- `mora` es textual, ocupa la última columna y no contiene faltantes.
- `tipo_credito`, `nivel_ahorro`, `deuda_sf`, `zona` y `saldo_pendiente_credito` están completos.

Fuentes de parametrización:

- [Banco ProCredit Ecuador — Informe de estrategias Q4 2023](https://www.bancoprocredit.com.ec/wp-content/uploads/2024/01/Informe-de-las-estrategias-de-Banco-ProCredit-cuarto-trimestre-2023-ok.pdf)
- [Ministerio del Trabajo — SBU 2026: USD 482](https://www.trabajo.gob.ec/despues-de-casi-una-decada-hay-consenso-gobierno-empleadores-y-trabajadores-acuerdan-fijar-el-salario-basico-unificado-de-2026-en-usd-482-no-hay-imposicion-hay-union/)

## Configuración

In [1]:
# Línea 0001: Importa Path para construir rutas de archivos de forma compatible.
from pathlib import Path

# Línea 0002: Importa NumPy para generar números aleatorios y operar con arreglos.
import numpy as np
# Línea 0003: Importa pandas para organizar, validar y exportar la base de datos.
import pandas as pd


# Línea 0004: Calcula o asigna el contenido de `N_FILAS` para utilizarlo después.
N_FILAS = 21_500
# Línea 0005: Calcula o asigna el contenido de `SEMILLA` para utilizarlo después.
SEMILLA = 20260903
# Línea 0006: Calcula o asigna el contenido de `SBU_2026_USD` para utilizarlo después.
SBU_2026_USD = 482.0
# Línea 0007: Calcula o asigna el contenido de `DESCARGAR_EN_COLAB` para utilizarlo después.
DESCARGAR_EN_COLAB = True
# Línea 0008: Calcula o asigna el contenido de `NOMBRE_SALIDA` para utilizarlo después.
NOMBRE_SALIDA = "dataset_mora.csv"


# Línea 0009: Calcula o asigna el contenido de `ETIQUETAS_MORA` para utilizarlo después.
ETIQUETAS_MORA = {
    # Línea 0010: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    0: "AL DÍA",
    # Línea 0011: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    1: "MORA DE 1 A 7 DÍAS",
    # Línea 0012: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    2: "MORA DE 8 A 30 DÍAS",
    # Línea 0013: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    3: "MORA DE 31 A 60 DÍAS",
    # Línea 0014: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    4: "MORA DE 61 A 90 DÍAS",
    # Línea 0015: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    5: "MORA DE MÁS DE 90 DÍAS",
# Línea 0016: Cierra la estructura, llamada o colección iniciada previamente.
}


# Línea 0017: Calcula o asigna el contenido de `NIVEL_MORA_POR_TEXTO` para utilizarlo después.
NIVEL_MORA_POR_TEXTO = {texto: nivel for nivel, texto in ETIQUETAS_MORA.items()}


# Línea 0018: Calcula o asigna el contenido de `COLUMNAS_COMPLETAS` para utilizarlo después.
COLUMNAS_COMPLETAS = {
    # Línea 0019: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "ID",
    # Línea 0020: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "tipo_credito",
    # Línea 0021: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "nivel_ahorro",
    # Línea 0022: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "deuda_sf",
    # Línea 0023: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "zona",
    # Línea 0024: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "saldo_pendiente_credito",
    # Línea 0025: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "mora",
# Línea 0026: Cierra la estructura, llamada o colección iniciada previamente.
}


# Tasas base de ausencia por columna. Se aplican de forma reproducible.
# Algunas variables derivadas también quedan ausentes si falta un insumo.
# Línea 0027: Calcula o asigna el contenido de `TASAS_FALTANTES` para utilizarlo después.
TASAS_FALTANTES = {
    # Línea 0028: Registra el valor correspondiente a `atraso` dentro de la configuración.
    "atraso": 0.020,
    # Línea 0029: Registra el valor correspondiente a `vivienda` dentro de la configuración.
    "vivienda": 0.040,
    # Línea 0030: Registra el valor correspondiente a `edad` dentro de la configuración.
    "edad": 0.010,
    # Línea 0031: Registra el valor correspondiente a `dias_lab` dentro de la configuración.
    "dias_lab": 0.060,
    # Línea 0032: Registra el valor correspondiente a `exp_sf` dentro de la configuración.
    "exp_sf": 0.080,
    # Línea 0033: Registra el valor correspondiente a `nivel_ahorro` dentro de la configuración.
    "nivel_ahorro": 0.000,
    # Línea 0034: Registra el valor correspondiente a `ingreso` dentro de la configuración.
    "ingreso": 0.040,
    # Línea 0035: Registra el valor correspondiente a `linea_sf` dentro de la configuración.
    "linea_sf": 0.090,
    # Línea 0036: Registra el valor correspondiente a `deuda_sf` dentro de la configuración.
    "deuda_sf": 0.000,
    # Línea 0037: Registra el valor correspondiente a `score` dentro de la configuración.
    "score": 0.030,
    # Línea 0038: Registra el valor correspondiente a `zona` dentro de la configuración.
    "zona": 0.000,
    # Línea 0039: Registra el valor correspondiente a `clasif_sbs` dentro de la configuración.
    "clasif_sbs": 0.040,
    # Línea 0040: Registra el valor correspondiente a `nivel_educ` dentro de la configuración.
    "nivel_educ": 0.050,
    # Línea 0041: Registra el valor correspondiente a `mora_max_3m` dentro de la configuración.
    "mora_max_3m": 0.050,
    # Línea 0042: Registra el valor correspondiente a `mora_max_6m` dentro de la configuración.
    "mora_max_6m": 0.060,
    # Línea 0043: Registra el valor correspondiente a `mora_max_12m` dentro de la configuración.
    "mora_max_12m": 0.070,
    # Línea 0044: Registra el valor correspondiente a `atrasos_6m_12m` dentro de la configuración.
    "atrasos_6m_12m": 0.060,
    # Línea 0045: Registra el valor correspondiente a `atrasos_12m` dentro de la configuración.
    "atrasos_12m": 0.070,
    # Línea 0046: Registra el valor correspondiente a `dti` dentro de la configuración.
    "dti": 0.050,
    # Línea 0047: Registra el valor correspondiente a `cuota_ingreso` dentro de la configuración.
    "cuota_ingreso": 0.060,
    # Línea 0048: Registra el valor correspondiente a `saldo_promedio_3m` dentro de la configuración.
    "saldo_promedio_3m": 0.090,
    # Línea 0049: Registra el valor correspondiente a `variacion_depositos_ingresos` dentro de la configuración.
    "variacion_depositos_ingresos": 0.120,
    # Línea 0050: Registra el valor correspondiente a `num_entidades_deuda` dentro de la configuración.
    "num_entidades_deuda": 0.050,
    # Línea 0051: Registra el valor correspondiente a `nuevos_creditos_3m_6m` dentro de la configuración.
    "nuevos_creditos_3m_6m": 0.070,
    # Línea 0052: Registra el valor correspondiente a `nuevos_creditos_6m` dentro de la configuración.
    "nuevos_creditos_6m": 0.070,
    # Línea 0053: Registra el valor correspondiente a `porcentaje_utilizacion_linea` dentro de la configuración.
    "porcentaje_utilizacion_linea": 0.060,
    # Línea 0054: Registra el valor correspondiente a `saldo_pendiente_credito` dentro de la configuración.
    "saldo_pendiente_credito": 0.000,
    # Línea 0055: Registra el valor correspondiente a `cuotas_restantes` dentro de la configuración.
    "cuotas_restantes": 0.040,
    # Línea 0056: Registra el valor correspondiente a `tipo_credito` dentro de la configuración.
    "tipo_credito": 0.000,
    # Línea 0057: Registra el valor correspondiente a `antiguedad_cliente_banco` dentro de la configuración.
    "antiguedad_cliente_banco": 0.060,
    # Línea 0058: Registra el valor correspondiente a `numero_productos_banco` dentro de la configuración.
    "numero_productos_banco": 0.040,
    # Línea 0059: Registra el valor correspondiente a `reestructuraciones_refinanciamientos_previos` dentro de la configuración.
    "reestructuraciones_refinanciamientos_previos": 0.030,
# Línea 0060: Cierra la estructura, llamada o colección iniciada previamente.
}


# Línea 0061: Calcula o asigna el contenido de `COLUMNAS` para utilizarlo después.
COLUMNAS = [
    # Línea 0062: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "ID",
    # Línea 0063: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "atraso",
    # Línea 0064: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "vivienda",
    # Línea 0065: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "edad",
    # Línea 0066: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "dias_lab",
    # Línea 0067: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "exp_sf",
    # Línea 0068: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "nivel_ahorro",
    # Línea 0069: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "ingreso",
    # Línea 0070: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "linea_sf",
    # Línea 0071: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "deuda_sf",
    # Línea 0072: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "score",
    # Línea 0073: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "zona",
    # Línea 0074: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "clasif_sbs",
    # Línea 0075: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "nivel_educ",
    # Línea 0076: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "mora_max_3m",
    # Línea 0077: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "mora_max_6m",
    # Línea 0078: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "mora_max_12m",
    # Línea 0079: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "atrasos_6m_12m",
    # Línea 0080: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "atrasos_12m",
    # Línea 0081: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "dti",
    # Línea 0082: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "cuota_ingreso",
    # Línea 0083: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "saldo_promedio_3m",
    # Línea 0084: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "variacion_depositos_ingresos",
    # Línea 0085: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "num_entidades_deuda",
    # Línea 0086: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "nuevos_creditos_3m_6m",
    # Línea 0087: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "nuevos_creditos_6m",
    # Línea 0088: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "porcentaje_utilizacion_linea",
    # Línea 0089: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "saldo_pendiente_credito",
    # Línea 0090: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "cuotas_restantes",
    # Línea 0091: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "tipo_credito",
    # Línea 0092: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "antiguedad_cliente_banco",
    # Línea 0093: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "numero_productos_banco",
    # Línea 0094: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "reestructuraciones_refinanciamientos_previos",
    # Línea 0095: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "mora",
# Línea 0096: Cierra la estructura, llamada o colección iniciada previamente.
]


# Línea 0097: Calcula o asigna el contenido de `COLUMNAS_ENTERAS_CON_FALTANTES` para utilizarlo después.
COLUMNAS_ENTERAS_CON_FALTANTES = [
    # Línea 0098: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "atraso",
    # Línea 0099: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "edad",
    # Línea 0100: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "dias_lab",
    # Línea 0101: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "exp_sf",
    # Línea 0102: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "score",
    # Línea 0103: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "mora_max_3m",
    # Línea 0104: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "mora_max_6m",
    # Línea 0105: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "mora_max_12m",
    # Línea 0106: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "atrasos_6m_12m",
    # Línea 0107: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "atrasos_12m",
    # Línea 0108: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "num_entidades_deuda",
    # Línea 0109: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "nuevos_creditos_3m_6m",
    # Línea 0110: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "nuevos_creditos_6m",
    # Línea 0111: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "cuotas_restantes",
    # Línea 0112: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "antiguedad_cliente_banco",
    # Línea 0113: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "numero_productos_banco",
# Línea 0114: Cierra la estructura, llamada o colección iniciada previamente.
]


# Línea 0115: Calcula o asigna el contenido de `COLUMNAS_MONETARIAS` para utilizarlo después.
COLUMNAS_MONETARIAS = [
    # Línea 0116: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "ingreso",
    # Línea 0117: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "linea_sf",
    # Línea 0118: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "deuda_sf",
    # Línea 0119: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "saldo_promedio_3m",
    # Línea 0120: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "saldo_pendiente_credito",
# Línea 0121: Cierra la estructura, llamada o colección iniciada previamente.
]


# Línea 0122: Calcula o asigna el contenido de `COLUMNAS_PORCENTAJE` para utilizarlo después.
COLUMNAS_PORCENTAJE = [
    # Línea 0123: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "dti",
    # Línea 0124: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "cuota_ingreso",
    # Línea 0125: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "variacion_depositos_ingresos",
    # Línea 0126: Ejecuta esta instrucción como parte del cálculo o validación descritos.
    "porcentaje_utilizacion_linea",
# Línea 0127: Cierra la estructura, llamada o colección iniciada previamente.
]

## Funciones auxiliares

In [2]:
# Línea 0001: Define la función `_sigmoide` para encapsular esta parte del proceso.
def _sigmoide(x):
    # Línea 0002: Devuelve el resultado calculado por la función.
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))


# Línea 0003: Define la función `_muestrear_por_fila` para encapsular esta parte del proceso.
def _muestrear_por_fila(rng, categorias, probabilidades):
    # Línea 0004: Documenta en español la finalidad de la función o del bloque.
    """Muestrea una categoría por fila desde una matriz de probabilidades."""
    # Línea 0005: Calcula o asigna el contenido de `probabilidades` para utilizarlo después.
    probabilidades = np.asarray(probabilidades, dtype=float)
    # Línea 0006: Calcula o asigna el contenido de `probabilidades` para utilizarlo después.
    probabilidades = probabilidades / probabilidades.sum(axis=1, keepdims=True)
    # Línea 0007: Calcula o asigna el contenido de `acumuladas` para utilizarlo después.
    acumuladas = np.cumsum(probabilidades, axis=1)
    # Línea 0008: Calcula o asigna el contenido de `u` para utilizarlo después.
    u = rng.random(probabilidades.shape[0])[:, None]
    # Línea 0009: Calcula o asigna el contenido de `indices` para utilizarlo después.
    indices = (u > acumuladas).sum(axis=1)
    # Línea 0010: Devuelve el resultado calculado por la función.
    return np.asarray(categorias, dtype=object)[indices]


# Línea 0011: Define la función `_dias_segun_clase` para encapsular esta parte del proceso.
def _dias_segun_clase(rng, clases):
    # Línea 0012: Documenta en español la finalidad de la función o del bloque.
    """Convierte clases de mora 0-5 a días dentro de sus intervalos."""
    # Línea 0013: Calcula o asigna el contenido de `clases` para utilizarlo después.
    clases = np.asarray(clases, dtype=int)
    # Línea 0014: Calcula o asigna el contenido de `dias` para utilizarlo después.
    dias = np.zeros(len(clases), dtype=int)
    # Línea 0015: Calcula o asigna el contenido de `rangos` para utilizarlo después.
    rangos = {1: (1, 7), 2: (8, 30), 3: (31, 60), 4: (61, 90), 5: (91, 240)}
    # Línea 0016: Recorre cada elemento requerido para aplicar la operación de forma repetida.
    for clase, (minimo, maximo) in rangos.items():
        # Línea 0017: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        mascara = clases == clase
        # Línea 0018: Calcula o asigna el contenido de `dias[mascara]` para utilizarlo después.
        dias[mascara] = rng.integers(minimo, maximo + 1, mascara.sum())
    # Línea 0019: Devuelve el resultado calculado por la función.
    return dias


# Línea 0020: Define la función `_asignar_mora_ordenada` para encapsular esta parte del proceso.
def _asignar_mora_ordenada(riesgo):
    # Línea 0021: Documenta en español la finalidad de la función o del bloque.
    """Asigna una mora desbalanceada y ordenada a partir del riesgo latente."""
    # Línea 0022: Calcula o asigna el contenido de `proporciones` para utilizarlo después.
    proporciones = np.array([0.720, 0.100, 0.070, 0.045, 0.035, 0.030])
    # Línea 0023: Calcula o asigna el contenido de `conteos` para utilizarlo después.
    conteos = np.floor(proporciones * len(riesgo)).astype(int)
    # Línea 0024: Calcula o asigna el contenido de `conteos[0] +` para utilizarlo después.
    conteos[0] += len(riesgo) - conteos.sum()
    # Línea 0025: Calcula o asigna el contenido de `orden` para utilizarlo después.
    orden = np.argsort(riesgo, kind="mergesort")
    # Línea 0026: Calcula o asigna el contenido de `mora` para utilizarlo después.
    mora = np.empty(len(riesgo), dtype=np.int8)
    # Línea 0027: Calcula o asigna el contenido de `inicio` para utilizarlo después.
    inicio = 0
    # Línea 0028: Recorre cada elemento requerido para aplicar la operación de forma repetida.
    for clase, cantidad in enumerate(conteos):
        # Línea 0029: Calcula o asigna el contenido de `seleccion` para utilizarlo después.
        seleccion = orden[inicio : inicio + cantidad]
        # Línea 0030: Calcula o asigna el contenido de `mora[seleccion]` para utilizarlo después.
        mora[seleccion] = clase
        # Línea 0031: Calcula o asigna el contenido de `inicio +` para utilizarlo después.
        inicio += cantidad
    # Línea 0032: Devuelve el resultado calculado por la función.
    return mora


# Línea 0033: Define la función `_clasificacion_sbs` para encapsular esta parte del proceso.
def _clasificacion_sbs(rng, mora):
    # Línea 0034: Documenta en español la finalidad de la función o del bloque.
    """Genera una clasificación A1-E compatible con cada tramo de mora."""
    # Línea 0035: Calcula o asigna el contenido de `opciones` para utilizarlo después.
    opciones = {
        # Línea 0036: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        0: (np.array(["A1", "A2", "A3"]), [0.50, 0.32, 0.18]),
        # Línea 0037: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        1: (np.array(["A3", "B1"]), [0.45, 0.55]),
        # Línea 0038: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        2: (np.array(["B1", "B2"]), [0.30, 0.70]),
        # Línea 0039: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        3: (np.array(["C1", "C2"]), [0.38, 0.62]),
        # Línea 0040: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        4: (np.array(["D"]), [1.00]),
        # Línea 0041: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        5: (np.array(["E"]), [1.00]),
    # Línea 0042: Cierra la estructura, llamada o colección iniciada previamente.
    }
    # Línea 0043: Calcula o asigna el contenido de `resultado` para utilizarlo después.
    resultado = np.empty(len(mora), dtype=object)
    # Línea 0044: Recorre cada elemento requerido para aplicar la operación de forma repetida.
    for clase, (categorias, probabilidades) in opciones.items():
        # Línea 0045: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        mascara = mora == clase
        # Línea 0046: Calcula o asigna el contenido de `resultado[mascara]` para utilizarlo después.
        resultado[mascara] = rng.choice(
            # Línea 0047: Calcula o asigna el contenido de `categorias, size` para utilizarlo después.
            categorias, size=mascara.sum(), p=probabilidades
        # Línea 0048: Cierra la estructura, llamada o colección iniciada previamente.
        )
    # Línea 0049: Devuelve el resultado calculado por la función.
    return resultado


# Línea 0050: Define la función `_digito_verificador_cedula` para encapsular esta parte del proceso.
def _digito_verificador_cedula(primeros_nueve):
    # Línea 0051: Documenta en español la finalidad de la función o del bloque.
    """Calcula el décimo dígito de la cédula ecuatoriana (módulo 10)."""
    # Línea 0052: Calcula o asigna el contenido de `coeficientes` para utilizarlo después.
    coeficientes = (2, 1, 2, 1, 2, 1, 2, 1, 2)
    # Línea 0053: Calcula o asigna el contenido de `total` para utilizarlo después.
    total = 0
    # Línea 0054: Recorre cada elemento requerido para aplicar la operación de forma repetida.
    for digito, coeficiente in zip(map(int, primeros_nueve), coeficientes):
        # Línea 0055: Calcula o asigna el contenido de `producto` para utilizarlo después.
        producto = digito * coeficiente
        # Línea 0056: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        total += producto - 9 if producto >= 10 else producto
    # Línea 0057: Devuelve el resultado calculado por la función.
    return (10 - total % 10) % 10


# Línea 0058: Define la función `_generar_cedulas_ecuatorianas` para encapsular esta parte del proceso.
def _generar_cedulas_ecuatorianas(rng, zonas):
    # Línea 0059: Documenta en español la finalidad de la función o del bloque.
    """Genera cédulas únicas y válidas, alineando provincia de emisión y zona."""
    # Línea 0060: Calcula o asigna el contenido de `codigo_provincia` para utilizarlo después.
    codigo_provincia = {
        # Línea 0061: Registra el valor correspondiente a `CUENCA` dentro de la configuración.
        "CUENCA": "01",
        # Línea 0062: Registra el valor correspondiente a `GUAYAQUIL` dentro de la configuración.
        "GUAYAQUIL": "09",
        # Línea 0063: Registra el valor correspondiente a `IBARRA` dentro de la configuración.
        "IBARRA": "10",
        # Línea 0064: Registra el valor correspondiente a `QUITO` dentro de la configuración.
        "QUITO": "17",
        # Línea 0065: Registra el valor correspondiente a `AMBATO` dentro de la configuración.
        "AMBATO": "18",
        # Línea 0066: Registra el valor correspondiente a `SANTO DOMINGO` dentro de la configuración.
        "SANTO DOMINGO": "23",
    # Línea 0067: Cierra la estructura, llamada o colección iniciada previamente.
    }
    # Línea 0068: Calcula o asigna el contenido de `cedulas` para utilizarlo después.
    cedulas = []
    # Línea 0069: Calcula o asigna el contenido de `usadas` para utilizarlo después.
    usadas = set()
    # Línea 0070: Recorre cada elemento requerido para aplicar la operación de forma repetida.
    for zona in zonas:
        # Línea 0071: Repite el bloque hasta obtener un valor que cumpla las condiciones.
        while True:
            # Línea 0072: Calcula o asigna el contenido de `tercer_digito` para utilizarlo después.
            tercer_digito = int(rng.integers(0, 6))
            # Línea 0073: Calcula o asigna el contenido de `consecutivo` para utilizarlo después.
            consecutivo = int(rng.integers(0, 1_000_000))
            # Línea 0074: Calcula o asigna el contenido de `primeros_nueve` para utilizarlo después.
            primeros_nueve = (
                # Línea 0075: Ejecuta esta instrucción como parte del cálculo o validación descritos.
                f"{codigo_provincia[zona]}{tercer_digito}{consecutivo:06d}"
            # Línea 0076: Cierra la estructura, llamada o colección iniciada previamente.
            )
            # Línea 0077: Calcula o asigna el contenido de `cedula` para utilizarlo después.
            cedula = primeros_nueve + str(
                # Línea 0078: Ejecuta esta instrucción como parte del cálculo o validación descritos.
                _digito_verificador_cedula(primeros_nueve)
            # Línea 0079: Cierra la estructura, llamada o colección iniciada previamente.
            )
            # Línea 0080: Evalúa la condición y ejecuta el bloque únicamente cuando es verdadera.
            if cedula not in usadas:
                # Línea 0081: Ejecuta esta instrucción como parte del cálculo o validación descritos.
                usadas.add(cedula)
                # Línea 0082: Ejecuta esta instrucción como parte del cálculo o validación descritos.
                cedulas.append(cedula)
                # Línea 0083: Ejecuta esta instrucción como parte del cálculo o validación descritos.
                break
    # Línea 0084: Devuelve el resultado calculado por la función.
    return np.asarray(cedulas, dtype=object)


# Línea 0085: Define la función `_validar_cedula_ecuatoriana` para encapsular esta parte del proceso.
def _validar_cedula_ecuatoriana(cedula):
    # Línea 0086: Documenta en español la finalidad de la función o del bloque.
    """Valida longitud, provincia, persona natural y dígito verificador."""
    # Línea 0087: Calcula o asigna el contenido de `texto` para utilizarlo después.
    texto = str(cedula).zfill(10)
    # Línea 0088: Evalúa la condición y ejecuta el bloque únicamente cuando es verdadera.
    if len(texto) != 10 or not texto.isdigit():
        # Línea 0089: Devuelve el resultado calculado por la función.
        return False
    # Línea 0090: Calcula o asigna el contenido de `provincia` para utilizarlo después.
    provincia = int(texto[:2])
    # Línea 0091: Evalúa la condición y ejecuta el bloque únicamente cuando es verdadera.
    if not 1 <= provincia <= 24 or int(texto[2]) > 5:
        # Línea 0092: Devuelve el resultado calculado por la función.
        return False
    # Línea 0093: Devuelve el resultado calculado por la función.
    return int(texto[-1]) == _digito_verificador_cedula(texto[:9])

## Generador de datos

In [3]:
# Línea 0001: Define la función `generar_base` para encapsular esta parte del proceso.
def generar_base(n_filas=N_FILAS, semilla=SEMILLA):
    # Línea 0002: Calcula o asigna el contenido de `rng` para utilizarlo después.
    rng = np.random.default_rng(semilla)

    # Ciudades y participación reportadas por Banco ProCredit Ecuador (Q4 2023).
    # Línea 0003: Calcula o asigna el contenido de `zonas_procredit` para utilizarlo después.
    zonas_procredit = np.array(
        # Línea 0004: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        ["QUITO", "GUAYAQUIL", "AMBATO", "IBARRA", "CUENCA", "SANTO DOMINGO"]
    # Línea 0005: Cierra la estructura, llamada o colección iniciada previamente.
    )
    # Línea 0006: Calcula o asigna el contenido de `pesos_zonas` para utilizarlo después.
    pesos_zonas = np.array([55.69, 22.49, 12.93, 7.17, 0.89, 0.82])
    # Línea 0007: Calcula o asigna el contenido de `pesos_zonas /` para utilizarlo después.
    pesos_zonas /= pesos_zonas.sum()
    # Línea 0008: Calcula o asigna el contenido de `zona` para utilizarlo después.
    zona = rng.choice(zonas_procredit, n_filas, p=pesos_zonas)

    # Línea 0009: Calcula o asigna el contenido de `edad` para utilizarlo después.
    edad = np.rint(rng.normal(39, 11.5, n_filas)).astype(int)
    # Línea 0010: Calcula o asigna el contenido de `edad` para utilizarlo después.
    edad = np.clip(edad, 18, 78)

    # Línea 0011: Calcula o asigna el contenido de `niveles_educ` para utilizarlo después.
    niveles_educ = np.array(
        # Línea 0012: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        [
            # Línea 0013: Ejecuta esta instrucción como parte del cálculo o validación descritos.
            "SIN INSTRUCCIÓN",
            # Línea 0014: Ejecuta esta instrucción como parte del cálculo o validación descritos.
            "PRIMARIA",
            # Línea 0015: Ejecuta esta instrucción como parte del cálculo o validación descritos.
            "SECUNDARIA",
            # Línea 0016: Ejecuta esta instrucción como parte del cálculo o validación descritos.
            "TÉCNICO/TECNOLÓGICO",
            # Línea 0017: Ejecuta esta instrucción como parte del cálculo o validación descritos.
            "UNIVERSITARIA",
            # Línea 0018: Ejecuta esta instrucción como parte del cálculo o validación descritos.
            "POSGRADO",
        # Línea 0019: Cierra la estructura, llamada o colección iniciada previamente.
        ]
    # Línea 0020: Cierra la estructura, llamada o colección iniciada previamente.
    )
    # Línea 0021: Calcula o asigna el contenido de `nivel_educ` para utilizarlo después.
    nivel_educ = rng.choice(
        # Línea 0022: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        niveles_educ,
        # Línea 0023: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        n_filas,
        # Línea 0024: Calcula o asigna el contenido de `p` para utilizarlo después.
        p=[0.015, 0.180, 0.420, 0.170, 0.190, 0.025],
    # Línea 0025: Cierra la estructura, llamada o colección iniciada previamente.
    )

    # Línea 0026: Calcula o asigna el contenido de `factor_educacion` para utilizarlo después.
    factor_educacion = {
        # Línea 0027: Registra el valor correspondiente a `SIN INSTRUCCIÓN` dentro de la configuración.
        "SIN INSTRUCCIÓN": 0.78,
        # Línea 0028: Registra el valor correspondiente a `PRIMARIA` dentro de la configuración.
        "PRIMARIA": 0.92,
        # Línea 0029: Registra el valor correspondiente a `SECUNDARIA` dentro de la configuración.
        "SECUNDARIA": 1.18,
        # Línea 0030: Registra el valor correspondiente a `TÉCNICO/TECNOLÓGICO` dentro de la configuración.
        "TÉCNICO/TECNOLÓGICO": 1.48,
        # Línea 0031: Registra el valor correspondiente a `UNIVERSITARIA` dentro de la configuración.
        "UNIVERSITARIA": 2.05,
        # Línea 0032: Registra el valor correspondiente a `POSGRADO` dentro de la configuración.
        "POSGRADO": 3.00,
    # Línea 0033: Cierra la estructura, llamada o colección iniciada previamente.
    }
    # Línea 0034: Calcula o asigna el contenido de `factor_provincia` para utilizarlo después.
    factor_provincia = {
        # Línea 0035: Registra el valor correspondiente a `QUITO` dentro de la configuración.
        "QUITO": 1.15,
        # Línea 0036: Registra el valor correspondiente a `GUAYAQUIL` dentro de la configuración.
        "GUAYAQUIL": 1.08,
        # Línea 0037: Registra el valor correspondiente a `AMBATO` dentro de la configuración.
        "AMBATO": 1.02,
        # Línea 0038: Registra el valor correspondiente a `IBARRA` dentro de la configuración.
        "IBARRA": 0.98,
        # Línea 0039: Registra el valor correspondiente a `CUENCA` dentro de la configuración.
        "CUENCA": 1.08,
        # Línea 0040: Registra el valor correspondiente a `SANTO DOMINGO` dentro de la configuración.
        "SANTO DOMINGO": 0.97,
    # Línea 0041: Cierra la estructura, llamada o colección iniciada previamente.
    }
    # Línea 0042: Calcula o asigna el contenido de `f_educ` para utilizarlo después.
    f_educ = np.array([factor_educacion[x] for x in nivel_educ])
    # Línea 0043: Calcula o asigna el contenido de `f_prov` para utilizarlo después.
    f_prov = np.array([factor_provincia[x] for x in zona])
    # Línea 0044: Calcula o asigna el contenido de `f_edad` para utilizarlo después.
    f_edad = np.clip(0.82 + 0.018 * (edad - 18), 0.82, 1.38)
    # Línea 0045: Calcula o asigna el contenido de `ingreso` para utilizarlo después.
    ingreso = SBU_2026_USD * f_educ * f_prov * f_edad
    # Línea 0046: Calcula o asigna el contenido de `ingreso *` para utilizarlo después.
    ingreso *= rng.lognormal(mean=-0.12, sigma=0.50, size=n_filas)
    # Línea 0047: Calcula o asigna el contenido de `ingreso` para utilizarlo después.
    ingreso = np.round(np.clip(ingreso, 300, 30_000), 2)

    # Línea 0048: Calcula o asigna el contenido de `prob_vivienda_propia` para utilizarlo después.
    prob_vivienda_propia = _sigmoide(
        # Línea 0049: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        -1.15
        # Línea 0050: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        + 0.045 * (edad - 35)
        # Línea 0051: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        + 0.34 * np.log1p(ingreso / SBU_2026_USD)
    # Línea 0052: Cierra la estructura, llamada o colección iniciada previamente.
    )
    # Línea 0053: Calcula o asigna el contenido de `vivienda` para utilizarlo después.
    vivienda = np.where(
        # Línea 0054: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        rng.random(n_filas) < prob_vivienda_propia, "PROPIA", "ARRENDADA"
    # Línea 0055: Cierra la estructura, llamada o colección iniciada previamente.
    )

    # Línea 0056: Calcula o asigna el contenido de `max_dias_laborables` para utilizarlo después.
    max_dias_laborables = np.maximum(0, (edad - 18) * 365)
    # Línea 0057: Calcula o asigna el contenido de `dias_lab` para utilizarlo después.
    dias_lab = np.floor(rng.beta(1.65, 3.1, n_filas) * max_dias_laborables).astype(int)
    # Línea 0058: Calcula o asigna el contenido de `exp_sf` para utilizarlo después.
    exp_sf = np.floor(rng.gamma(2.15, 34.0, n_filas)).astype(int)
    # Línea 0059: Calcula o asigna el contenido de `exp_sf` para utilizarlo después.
    exp_sf = np.minimum(exp_sf, np.maximum(0, (edad - 18) * 12))

    # Línea 0060: Calcula o asigna el contenido de `tipos_credito` para utilizarlo después.
    tipos_credito = np.array(
        # Línea 0061: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        [
            # Línea 0062: Ejecuta esta instrucción como parte del cálculo o validación descritos.
            "CONSUMO",
            # Línea 0063: Ejecuta esta instrucción como parte del cálculo o validación descritos.
            "VIVIENDA",
            # Línea 0064: Ejecuta esta instrucción como parte del cálculo o validación descritos.
            "VEHICULAR",
            # Línea 0065: Ejecuta esta instrucción como parte del cálculo o validación descritos.
            "PRODUCTIVO",
            # Línea 0066: Ejecuta esta instrucción como parte del cálculo o validación descritos.
            "COMERCIAL",
            # Línea 0067: Ejecuta esta instrucción como parte del cálculo o validación descritos.
            "MICROCRÉDITO",
            # Línea 0068: Ejecuta esta instrucción como parte del cálculo o validación descritos.
            "AGROPECUARIO",
            # Línea 0069: Ejecuta esta instrucción como parte del cálculo o validación descritos.
            "EDUCATIVO",
            # Línea 0070: Ejecuta esta instrucción como parte del cálculo o validación descritos.
            "TARJETA DE CRÉDITO",
        # Línea 0071: Cierra la estructura, llamada o colección iniciada previamente.
        ]
    # Línea 0072: Cierra la estructura, llamada o colección iniciada previamente.
    )
    # Línea 0073: Calcula o asigna el contenido de `tipo_credito` para utilizarlo después.
    tipo_credito = rng.choice(
        # Línea 0074: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        tipos_credito,
        # Línea 0075: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        n_filas,
        # Línea 0076: Calcula o asigna el contenido de `p` para utilizarlo después.
        p=[0.31, 0.10, 0.09, 0.10, 0.07, 0.15, 0.06, 0.03, 0.09],
    # Línea 0077: Cierra la estructura, llamada o colección iniciada previamente.
    )
    # Línea 0078: Calcula o asigna el contenido de `terminos` para utilizarlo después.
    terminos = {
        # Línea 0079: Registra el valor correspondiente a `CONSUMO` dentro de la configuración.
        "CONSUMO": (6, 60),
        # Línea 0080: Registra el valor correspondiente a `VIVIENDA` dentro de la configuración.
        "VIVIENDA": (60, 240),
        # Línea 0081: Registra el valor correspondiente a `VEHICULAR` dentro de la configuración.
        "VEHICULAR": (12, 84),
        # Línea 0082: Registra el valor correspondiente a `PRODUCTIVO` dentro de la configuración.
        "PRODUCTIVO": (12, 120),
        # Línea 0083: Registra el valor correspondiente a `COMERCIAL` dentro de la configuración.
        "COMERCIAL": (6, 84),
        # Línea 0084: Registra el valor correspondiente a `MICROCRÉDITO` dentro de la configuración.
        "MICROCRÉDITO": (3, 36),
        # Línea 0085: Registra el valor correspondiente a `AGROPECUARIO` dentro de la configuración.
        "AGROPECUARIO": (6, 60),
        # Línea 0086: Registra el valor correspondiente a `EDUCATIVO` dentro de la configuración.
        "EDUCATIVO": (6, 84),
        # Línea 0087: Registra el valor correspondiente a `TARJETA DE CRÉDITO` dentro de la configuración.
        "TARJETA DE CRÉDITO": (1, 48),
    # Línea 0088: Cierra la estructura, llamada o colección iniciada previamente.
    }
    # Línea 0089: Calcula o asigna el contenido de `factores_linea` para utilizarlo después.
    factores_linea = {
        # Línea 0090: Registra el valor correspondiente a `CONSUMO` dentro de la configuración.
        "CONSUMO": 6.0,
        # Línea 0091: Registra el valor correspondiente a `VIVIENDA` dentro de la configuración.
        "VIVIENDA": 45.0,
        # Línea 0092: Registra el valor correspondiente a `VEHICULAR` dentro de la configuración.
        "VEHICULAR": 18.0,
        # Línea 0093: Registra el valor correspondiente a `PRODUCTIVO` dentro de la configuración.
        "PRODUCTIVO": 25.0,
        # Línea 0094: Registra el valor correspondiente a `COMERCIAL` dentro de la configuración.
        "COMERCIAL": 20.0,
        # Línea 0095: Registra el valor correspondiente a `MICROCRÉDITO` dentro de la configuración.
        "MICROCRÉDITO": 5.0,
        # Línea 0096: Registra el valor correspondiente a `AGROPECUARIO` dentro de la configuración.
        "AGROPECUARIO": 12.0,
        # Línea 0097: Registra el valor correspondiente a `EDUCATIVO` dentro de la configuración.
        "EDUCATIVO": 10.0,
        # Línea 0098: Registra el valor correspondiente a `TARJETA DE CRÉDITO` dentro de la configuración.
        "TARJETA DE CRÉDITO": 4.0,
    # Línea 0099: Cierra la estructura, llamada o colección iniciada previamente.
    }
    # Línea 0100: Calcula o asigna el contenido de `tasas_anuales` para utilizarlo después.
    tasas_anuales = {
        # Línea 0101: Registra el valor correspondiente a `CONSUMO` dentro de la configuración.
        "CONSUMO": 0.160,
        # Línea 0102: Registra el valor correspondiente a `VIVIENDA` dentro de la configuración.
        "VIVIENDA": 0.090,
        # Línea 0103: Registra el valor correspondiente a `VEHICULAR` dentro de la configuración.
        "VEHICULAR": 0.140,
        # Línea 0104: Registra el valor correspondiente a `PRODUCTIVO` dentro de la configuración.
        "PRODUCTIVO": 0.120,
        # Línea 0105: Registra el valor correspondiente a `COMERCIAL` dentro de la configuración.
        "COMERCIAL": 0.130,
        # Línea 0106: Registra el valor correspondiente a `MICROCRÉDITO` dentro de la configuración.
        "MICROCRÉDITO": 0.220,
        # Línea 0107: Registra el valor correspondiente a `AGROPECUARIO` dentro de la configuración.
        "AGROPECUARIO": 0.140,
        # Línea 0108: Registra el valor correspondiente a `EDUCATIVO` dentro de la configuración.
        "EDUCATIVO": 0.100,
        # Línea 0109: Registra el valor correspondiente a `TARJETA DE CRÉDITO` dentro de la configuración.
        "TARJETA DE CRÉDITO": 0.180,
    # Línea 0110: Cierra la estructura, llamada o colección iniciada previamente.
    }

    # Línea 0111: Calcula o asigna el contenido de `cuotas_restantes` para utilizarlo después.
    cuotas_restantes = np.zeros(n_filas, dtype=int)
    # Línea 0112: Calcula o asigna el contenido de `factor_linea` para utilizarlo después.
    factor_linea = np.zeros(n_filas)
    # Línea 0113: Calcula o asigna el contenido de `tasa_anual` para utilizarlo después.
    tasa_anual = np.zeros(n_filas)
    # Línea 0114: Recorre cada elemento requerido para aplicar la operación de forma repetida.
    for tipo in tipos_credito:
        # Línea 0115: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        mascara = tipo_credito == tipo
        # Línea 0116: Calcula o asigna el contenido de `minimo, maximo` para utilizarlo después.
        minimo, maximo = terminos[tipo]
        # Línea 0117: Calcula o asigna el contenido de `cuotas_restantes[mascara]` para utilizarlo después.
        cuotas_restantes[mascara] = rng.integers(minimo, maximo + 1, mascara.sum())
        # Línea 0118: Calcula o asigna el contenido de `factor_linea[mascara]` para utilizarlo después.
        factor_linea[mascara] = factores_linea[tipo]
        # Línea 0119: Calcula o asigna el contenido de `tasa_anual[mascara]` para utilizarlo después.
        tasa_anual[mascara] = tasas_anuales[tipo]

    # Línea 0120: Calcula o asigna el contenido de `linea_sf` para utilizarlo después.
    linea_sf = ingreso * factor_linea * rng.lognormal(0.0, 0.48, n_filas)
    # Línea 0121: Calcula o asigna el contenido de `linea_sf` para utilizarlo después.
    linea_sf = np.round(np.clip(linea_sf, 300, 350_000), 2)

    # Línea 0122: Calcula o asigna el contenido de `num_entidades_deuda` para utilizarlo después.
    num_entidades_deuda = np.clip(1 + rng.poisson(1.05, n_filas), 1, 8)
    # Línea 0123: Calcula o asigna el contenido de `nuevos_creditos_6m` para utilizarlo después.
    nuevos_creditos_6m = np.clip(
        # Línea 0124: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        rng.poisson(0.22 + 0.09 * (num_entidades_deuda - 1), n_filas), 0, 6
    # Línea 0125: Cierra la estructura, llamada o colección iniciada previamente.
    )
    # Línea 0126: Calcula o asigna el contenido de `nuevos_creditos_3m_6m` para utilizarlo después.
    nuevos_creditos_3m_6m = np.array(
        # Línea 0127: Calcula o asigna el contenido de `[rng.integers(0, valor + 1) for valor in nuevos_creditos_6m], dtype` para utilizarlo después.
        [rng.integers(0, valor + 1) for valor in nuevos_creditos_6m], dtype=int
    # Línea 0128: Cierra la estructura, llamada o colección iniciada previamente.
    )

    # Línea 0129: Calcula o asigna el contenido de `ahorro_meses` para utilizarlo después.
    ahorro_meses = rng.lognormal(
        # Línea 0130: Calcula o asigna el contenido de `mean` para utilizarlo después.
        mean=-0.35 + 0.12 * np.log1p(f_educ), sigma=1.00, size=n_filas
    # Línea 0131: Cierra la estructura, llamada o colección iniciada previamente.
    )
    # Línea 0132: Calcula o asigna el contenido de `sin_ahorro` para utilizarlo después.
    sin_ahorro = rng.random(n_filas) < 0.14
    # Línea 0133: Calcula o asigna el contenido de `ahorro_meses[sin_ahorro]` para utilizarlo después.
    ahorro_meses[sin_ahorro] = 0.0
    # Línea 0134: Calcula o asigna el contenido de `saldo_promedio_3m` para utilizarlo después.
    saldo_promedio_3m = np.round(np.clip(ingreso * ahorro_meses, 0, 120_000), 2)
    # Línea 0135: Calcula o asigna el contenido de `ratio_ahorro` para utilizarlo después.
    ratio_ahorro = saldo_promedio_3m / ingreso
    # Línea 0136: Calcula o asigna el contenido de `nivel_ahorro` para utilizarlo después.
    nivel_ahorro = np.select(
        # Línea 0137: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        [
            # Línea 0138: Ejecuta esta instrucción como parte del cálculo o validación descritos.
            ratio_ahorro < 0.10,
            # Línea 0139: Ejecuta esta instrucción como parte del cálculo o validación descritos.
            ratio_ahorro < 1.00,
            # Línea 0140: Ejecuta esta instrucción como parte del cálculo o validación descritos.
            ratio_ahorro < 3.00,
            # Línea 0141: Ejecuta esta instrucción como parte del cálculo o validación descritos.
            ratio_ahorro < 6.00,
        # Línea 0142: Cierra la estructura, llamada o colección iniciada previamente.
        ],
        # Línea 0143: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        ["SIN AHORRO", "BAJO", "MEDIO", "ALTO"],
        # Línea 0144: Calcula o asigna el contenido de `default` para utilizarlo después.
        default="MUY ALTO",
    # Línea 0145: Cierra la estructura, llamada o colección iniciada previamente.
    )

    # Línea 0146: Calcula o asigna el contenido de `utilizacion_base` para utilizarlo después.
    utilizacion_base = rng.beta(1.9, 2.6, n_filas)
    # Línea 0147: Calcula o asigna el contenido de `presion_crediticia` para utilizarlo después.
    presion_crediticia = (
        # Línea 0148: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        0.12 * (num_entidades_deuda - 1)
        # Línea 0149: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        + 0.18 * nuevos_creditos_6m
        # Línea 0150: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        - 0.07 * np.log1p(ratio_ahorro)
    # Línea 0151: Cierra la estructura, llamada o colección iniciada previamente.
    )
    # Línea 0152: Calcula o asigna el contenido de `porcentaje_utilizacion_linea` para utilizarlo después.
    porcentaje_utilizacion_linea = np.clip(
        # Línea 0153: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        100 * utilizacion_base + 12 * presion_crediticia + rng.normal(0, 6, n_filas),
        # Línea 0154: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        1,
        # Línea 0155: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        135,
    # Línea 0156: Cierra la estructura, llamada o colección iniciada previamente.
    )
    # Línea 0157: Calcula o asigna el contenido de `deuda_sf` para utilizarlo después.
    deuda_sf = np.round(linea_sf * porcentaje_utilizacion_linea / 100, 2)
    # Línea 0158: Calcula o asigna el contenido de `saldo_pendiente_credito` para utilizarlo después.
    saldo_pendiente_credito = np.round(
        # Línea 0159: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        deuda_sf * rng.uniform(0.45, 1.00, n_filas), 2
    # Línea 0160: Cierra la estructura, llamada o colección iniciada previamente.
    )

    # Línea 0161: Calcula o asigna el contenido de `tasa_mensual` para utilizarlo después.
    tasa_mensual = tasa_anual / 12
    # Línea 0162: Calcula o asigna el contenido de `factor_pago` para utilizarlo después.
    factor_pago = (
        # Línea 0163: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        tasa_mensual * (1 + tasa_mensual) ** cuotas_restantes
        # Línea 0164: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        / ((1 + tasa_mensual) ** cuotas_restantes - 1)
    # Línea 0165: Cierra la estructura, llamada o colección iniciada previamente.
    )
    # Línea 0166: Calcula o asigna el contenido de `cuota_mensual` para utilizarlo después.
    cuota_mensual = saldo_pendiente_credito * factor_pago
    # Línea 0167: Calcula o asigna el contenido de `cuota_ingreso` para utilizarlo después.
    cuota_ingreso = np.clip(100 * cuota_mensual / ingreso, 0.1, 95.0)
    # Línea 0168: Calcula o asigna el contenido de `dti` para utilizarlo después.
    dti = np.clip(100 * deuda_sf / ingreso, 0.1, 5000.0)

    # Línea 0169: Calcula o asigna el contenido de `riesgo_preliminar` para utilizarlo después.
    riesgo_preliminar = (
        # Línea 0170: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        1.15 * (porcentaje_utilizacion_linea / 100 - 0.45)
        # Línea 0171: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        + 0.55 * np.log1p(dti / 100)
        # Línea 0172: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        + 0.85 * np.maximum(cuota_ingreso / 100 - 0.30, 0)
        # Línea 0173: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        + 0.20 * (num_entidades_deuda - 2)
        # Línea 0174: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        + 0.28 * nuevos_creditos_6m
        # Línea 0175: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        - 0.35 * np.log1p(ratio_ahorro)
        # Línea 0176: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        - 0.18 * np.log1p(ingreso / SBU_2026_USD)
        # Línea 0177: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        - 0.10 * np.log1p((dias_lab + 1) / 365)
        # Línea 0178: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        + rng.normal(0, 0.80, n_filas)
    # Línea 0179: Cierra la estructura, llamada o colección iniciada previamente.
    )
    # Línea 0180: Calcula o asigna el contenido de `prob_reestructura` para utilizarlo después.
    prob_reestructura = _sigmoide(-3.10 + 0.85 * riesgo_preliminar)
    # Línea 0181: Calcula o asigna el contenido de `reestructura` para utilizarlo después.
    reestructura = rng.random(n_filas) < prob_reestructura
    # Línea 0182: Calcula o asigna el contenido de `riesgo` para utilizarlo después.
    riesgo = riesgo_preliminar + 0.85 * reestructura + rng.normal(0, 0.30, n_filas)
    # Línea 0183: Calcula o asigna el contenido de `mora` para utilizarlo después.
    mora = _asignar_mora_ordenada(riesgo)
    # Línea 0184: Calcula o asigna el contenido de `atraso` para utilizarlo después.
    atraso = _dias_segun_clase(rng, mora)

    # Línea 0185: Calcula o asigna el contenido de `score` para utilizarlo después.
    score = np.rint(835 - 82 * riesgo + rng.normal(0, 42, n_filas)).astype(int)
    # Línea 0186: Calcula o asigna el contenido de `score` para utilizarlo después.
    score = np.clip(score, 300, 999)
    # Línea 0187: Calcula o asigna el contenido de `clasif_sbs` para utilizarlo después.
    clasif_sbs = _clasificacion_sbs(rng, mora)

    # Línea 0188: Calcula o asigna el contenido de `incremento_3m` para utilizarlo después.
    incremento_3m = rng.choice([-1, 0, 1], n_filas, p=[0.10, 0.74, 0.16])
    # Línea 0189: Calcula o asigna el contenido de `clase_hist_3m` para utilizarlo después.
    clase_hist_3m = np.maximum(mora, np.clip(mora + incremento_3m, 0, 5))
    # Línea 0190: Calcula o asigna el contenido de `clase_hist_6m` para utilizarlo después.
    clase_hist_6m = np.minimum(
        # Línea 0191: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        5, clase_hist_3m + (rng.random(n_filas) < (0.12 + 0.04 * mora))
    # Línea 0192: Cierra la estructura, llamada o colección iniciada previamente.
    )
    # Línea 0193: Calcula o asigna el contenido de `clase_hist_12m` para utilizarlo después.
    clase_hist_12m = np.minimum(
        # Línea 0194: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        5, clase_hist_6m + (rng.random(n_filas) < (0.16 + 0.04 * mora))
    # Línea 0195: Cierra la estructura, llamada o colección iniciada previamente.
    )
    # Línea 0196: Calcula o asigna el contenido de `mora_max_3m` para utilizarlo después.
    mora_max_3m = np.maximum(atraso, _dias_segun_clase(rng, clase_hist_3m))
    # Línea 0197: Calcula o asigna el contenido de `mora_max_6m` para utilizarlo después.
    mora_max_6m = np.maximum(mora_max_3m, _dias_segun_clase(rng, clase_hist_6m))
    # Línea 0198: Calcula o asigna el contenido de `mora_max_12m` para utilizarlo después.
    mora_max_12m = np.maximum(mora_max_6m, _dias_segun_clase(rng, clase_hist_12m))

    # El número de eventos aumenta de forma ordenada con el nivel de mora.
    # Línea 0199: Calcula o asigna el contenido de `min_atrasos_6m` para utilizarlo después.
    min_atrasos_6m = np.array([0, 1, 2, 3, 4, 5])[mora]
    # Línea 0200: Calcula o asigna el contenido de `max_atrasos_6m` para utilizarlo después.
    max_atrasos_6m = np.array([2, 4, 6, 8, 10, 12])[mora]
    # Línea 0201: Calcula o asigna el contenido de `atrasos_6m_12m` para utilizarlo después.
    atrasos_6m_12m = np.array(
        # Línea 0202: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        [
            # Línea 0203: Ejecuta esta instrucción como parte del cálculo o validación descritos.
            rng.integers(minimo, maximo + 1)
            # Línea 0204: Recorre cada elemento requerido para aplicar la operación de forma repetida.
            for minimo, maximo in zip(min_atrasos_6m, max_atrasos_6m)
        # Línea 0205: Cierra la estructura, llamada o colección iniciada previamente.
        ],
        # Línea 0206: Calcula o asigna el contenido de `dtype` para utilizarlo después.
        dtype=int,
    # Línea 0207: Cierra la estructura, llamada o colección iniciada previamente.
    )
    # Línea 0208: Calcula o asigna el contenido de `atrasos_12m` para utilizarlo después.
    atrasos_12m = np.clip(
        # Línea 0209: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        atrasos_6m_12m
        # Línea 0210: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        + rng.poisson(0.15 + 0.18 * clase_hist_12m, n_filas),
        # Línea 0211: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        0,
        # Línea 0212: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        24,
    # Línea 0213: Cierra la estructura, llamada o colección iniciada previamente.
    )

    # Variación absoluta positiva (%), menor en promedio cuando aumenta la mora.
    # Línea 0214: Calcula o asigna el contenido de `variacion_depositos_ingresos` para utilizarlo después.
    variacion_depositos_ingresos = np.clip(
        # Línea 0215: Calcula o asigna el contenido de `rng.lognormal(mean` para utilizarlo después.
        rng.lognormal(mean=2.35 - 0.12 * mora, sigma=0.60, size=n_filas),
        # Línea 0216: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        0.01,
        # Línea 0217: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        120,
    # Línea 0218: Cierra la estructura, llamada o colección iniciada previamente.
    )
    # Línea 0219: Calcula o asigna el contenido de `antiguedad_cliente_banco` para utilizarlo después.
    antiguedad_cliente_banco = np.floor(rng.gamma(1.9, 27.0, n_filas)).astype(int)
    # Línea 0220: Calcula o asigna el contenido de `antiguedad_cliente_banco` para utilizarlo después.
    antiguedad_cliente_banco = np.minimum(antiguedad_cliente_banco, exp_sf)
    # Línea 0221: Calcula o asigna el contenido de `numero_productos_banco` para utilizarlo después.
    numero_productos_banco = np.clip(
        # Línea 0222: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        1
        # Línea 0223: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        + rng.poisson(
            # Línea 0224: Ejecuta esta instrucción como parte del cálculo o validación descritos.
            0.65
            # Línea 0225: Ejecuta esta instrucción como parte del cálculo o validación descritos.
            + 0.006 * antiguedad_cliente_banco
            # Línea 0226: Ejecuta esta instrucción como parte del cálculo o validación descritos.
            + 0.16 * np.log1p(ingreso / SBU_2026_USD),
            # Línea 0227: Ejecuta esta instrucción como parte del cálculo o validación descritos.
            n_filas,
        # Línea 0228: Cierra la estructura, llamada o colección iniciada previamente.
        ),
        # Línea 0229: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        1,
        # Línea 0230: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        10,
    # Línea 0231: Cierra la estructura, llamada o colección iniciada previamente.
    )

    # Cédulas sintéticas válidas; se conservan como texto para no perder ceros.
    # Línea 0232: Calcula o asigna el contenido de `identificadores` para utilizarlo después.
    identificadores = _generar_cedulas_ecuatorianas(rng, zona)

    # Transforma el nivel numérico interno en la etiqueta textual solicitada.
    # Línea 0233: Calcula o asigna el contenido de `mora_texto` para utilizarlo después.
    mora_texto = np.array([ETIQUETAS_MORA[nivel] for nivel in mora], dtype=object)

    # Línea 0234: Calcula o asigna el contenido de `df` para utilizarlo después.
    df = pd.DataFrame(
        # Línea 0235: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        {
            # Línea 0236: Registra el valor correspondiente a `ID` dentro de la configuración.
            "ID": identificadores,
            # Línea 0237: Registra el valor correspondiente a `atraso` dentro de la configuración.
            "atraso": atraso,
            # Línea 0238: Registra el valor correspondiente a `vivienda` dentro de la configuración.
            "vivienda": vivienda,
            # Línea 0239: Registra el valor correspondiente a `edad` dentro de la configuración.
            "edad": edad,
            # Línea 0240: Registra el valor correspondiente a `dias_lab` dentro de la configuración.
            "dias_lab": dias_lab,
            # Línea 0241: Registra el valor correspondiente a `exp_sf` dentro de la configuración.
            "exp_sf": exp_sf,
            # Línea 0242: Registra el valor correspondiente a `nivel_ahorro` dentro de la configuración.
            "nivel_ahorro": nivel_ahorro,
            # Línea 0243: Registra el valor correspondiente a `ingreso` dentro de la configuración.
            "ingreso": ingreso,
            # Línea 0244: Registra el valor correspondiente a `linea_sf` dentro de la configuración.
            "linea_sf": linea_sf,
            # Línea 0245: Registra el valor correspondiente a `deuda_sf` dentro de la configuración.
            "deuda_sf": deuda_sf,
            # Línea 0246: Registra el valor correspondiente a `score` dentro de la configuración.
            "score": score,
            # Línea 0247: Registra el valor correspondiente a `zona` dentro de la configuración.
            "zona": zona,
            # Línea 0248: Registra el valor correspondiente a `clasif_sbs` dentro de la configuración.
            "clasif_sbs": clasif_sbs,
            # Línea 0249: Registra el valor correspondiente a `nivel_educ` dentro de la configuración.
            "nivel_educ": nivel_educ,
            # Línea 0250: Registra el valor correspondiente a `mora_max_3m` dentro de la configuración.
            "mora_max_3m": mora_max_3m,
            # Línea 0251: Registra el valor correspondiente a `mora_max_6m` dentro de la configuración.
            "mora_max_6m": mora_max_6m,
            # Línea 0252: Registra el valor correspondiente a `mora_max_12m` dentro de la configuración.
            "mora_max_12m": mora_max_12m,
            # Línea 0253: Registra el valor correspondiente a `atrasos_6m_12m` dentro de la configuración.
            "atrasos_6m_12m": atrasos_6m_12m,
            # Línea 0254: Registra el valor correspondiente a `atrasos_12m` dentro de la configuración.
            "atrasos_12m": atrasos_12m,
            # Línea 0255: Registra el valor correspondiente a `dti` dentro de la configuración.
            "dti": dti,
            # Línea 0256: Registra el valor correspondiente a `cuota_ingreso` dentro de la configuración.
            "cuota_ingreso": cuota_ingreso,
            # Línea 0257: Registra el valor correspondiente a `saldo_promedio_3m` dentro de la configuración.
            "saldo_promedio_3m": saldo_promedio_3m,
            # Línea 0258: Registra el valor correspondiente a `variacion_depositos_ingresos` dentro de la configuración.
            "variacion_depositos_ingresos": variacion_depositos_ingresos,
            # Línea 0259: Registra el valor correspondiente a `num_entidades_deuda` dentro de la configuración.
            "num_entidades_deuda": num_entidades_deuda,
            # Línea 0260: Registra el valor correspondiente a `nuevos_creditos_3m_6m` dentro de la configuración.
            "nuevos_creditos_3m_6m": nuevos_creditos_3m_6m,
            # Línea 0261: Registra el valor correspondiente a `nuevos_creditos_6m` dentro de la configuración.
            "nuevos_creditos_6m": nuevos_creditos_6m,
            # Línea 0262: Registra el valor correspondiente a `porcentaje_utilizacion_linea` dentro de la configuración.
            "porcentaje_utilizacion_linea": porcentaje_utilizacion_linea,
            # Línea 0263: Registra el valor correspondiente a `saldo_pendiente_credito` dentro de la configuración.
            "saldo_pendiente_credito": saldo_pendiente_credito,
            # Línea 0264: Registra el valor correspondiente a `cuotas_restantes` dentro de la configuración.
            "cuotas_restantes": cuotas_restantes,
            # Línea 0265: Registra el valor correspondiente a `tipo_credito` dentro de la configuración.
            "tipo_credito": tipo_credito,
            # Línea 0266: Registra el valor correspondiente a `antiguedad_cliente_banco` dentro de la configuración.
            "antiguedad_cliente_banco": antiguedad_cliente_banco,
            # Línea 0267: Registra el valor correspondiente a `numero_productos_banco` dentro de la configuración.
            "numero_productos_banco": numero_productos_banco,
            # Línea 0268: Registra el valor correspondiente a `reestructuraciones_refinanciamientos_previos` dentro de la configuración.
            "reestructuraciones_refinanciamientos_previos": np.where(
                # Línea 0269: Ejecuta esta instrucción como parte del cálculo o validación descritos.
                reestructura, "SÍ", "NO"
            # Línea 0270: Cierra la estructura, llamada o colección iniciada previamente.
            ),
            # Línea 0271: Registra el valor correspondiente a `mora` dentro de la configuración.
            "mora": mora_texto,
        # Línea 0272: Cierra la estructura, llamada o colección iniciada previamente.
        },
        # Línea 0273: Calcula o asigna el contenido de `columns` para utilizarlo después.
        columns=COLUMNAS,
    # Línea 0274: Cierra la estructura, llamada o colección iniciada previamente.
    )

    # Máscaras de ausencia. Se propagan a razones que dependen de otros campos.
    # Línea 0275: Calcula o asigna el contenido de `mascaras` para utilizarlo después.
    mascaras = {
        # Línea 0276: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        columna: rng.random(n_filas) < tasa
        # Línea 0277: Recorre cada elemento requerido para aplicar la operación de forma repetida.
        for columna, tasa in TASAS_FALTANTES.items()
    # Línea 0278: Cierra la estructura, llamada o colección iniciada previamente.
    }
    # Línea 0279: Calcula o asigna el contenido de `mascaras["dti"] |` para utilizarlo después.
    mascaras["dti"] |= mascaras["ingreso"] | mascaras["deuda_sf"]
    # Línea 0280: Calcula o asigna el contenido de `mascaras["cuota_ingreso"] |` para utilizarlo después.
    mascaras["cuota_ingreso"] |= (
        # Línea 0281: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        mascaras["ingreso"]
        # Línea 0282: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        | mascaras["saldo_pendiente_credito"]
        # Línea 0283: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        | mascaras["cuotas_restantes"]
    # Línea 0284: Cierra la estructura, llamada o colección iniciada previamente.
    )
    # Línea 0285: Calcula o asigna el contenido de `mascaras["porcentaje_utilizacion_linea"] |` para utilizarlo después.
    mascaras["porcentaje_utilizacion_linea"] |= (
        # Línea 0286: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        mascaras["linea_sf"] | mascaras["deuda_sf"]
    # Línea 0287: Cierra la estructura, llamada o colección iniciada previamente.
    )

    # Línea 0288: Recorre cada elemento requerido para aplicar la operación de forma repetida.
    for columna, mascara in mascaras.items():
        # Línea 0289: Calcula o asigna el contenido de `df.loc[mascara, columna]` para utilizarlo después.
        df.loc[mascara, columna] = pd.NA

    # Tipos anulables: permiten enteros limpios con celdas vacías en el CSV.
    # Línea 0290: Calcula o asigna el contenido de `df["mora"]` para utilizarlo después.
    df["mora"] = df["mora"].astype("string")
    # Línea 0291: Calcula o asigna el contenido de `df["ID"]` para utilizarlo después.
    df["ID"] = df["ID"].astype("string")
    # Línea 0292: Recorre cada elemento requerido para aplicar la operación de forma repetida.
    for columna in COLUMNAS_ENTERAS_CON_FALTANTES:
        # Línea 0293: Calcula o asigna el contenido de `df[columna]` para utilizarlo después.
        df[columna] = pd.to_numeric(df[columna], errors="coerce").astype("Int64")
    # Línea 0294: Recorre cada elemento requerido para aplicar la operación de forma repetida.
    for columna in COLUMNAS_MONETARIAS + COLUMNAS_PORCENTAJE:
        # Línea 0295: Calcula o asigna el contenido de `df[columna]` para utilizarlo después.
        df[columna] = pd.to_numeric(df[columna], errors="coerce").astype("Float64")
    # Línea 0296: Recorre cada elemento requerido para aplicar la operación de forma repetida.
    for columna in set(COLUMNAS) - set(COLUMNAS_ENTERAS_CON_FALTANTES) - {
        # Línea 0297: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        "mora",
        # Línea 0298: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        "ID",
        # Línea 0299: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        *COLUMNAS_MONETARIAS,
        # Línea 0300: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        *COLUMNAS_PORCENTAJE,
    # Línea 0301: Cierra la estructura, llamada o colección iniciada previamente.
    }:
        # Línea 0302: Calcula o asigna el contenido de `df[columna]` para utilizarlo después.
        df[columna] = df[columna].astype("string")

    # Línea 0303: Calcula o asigna el contenido de `df[COLUMNAS_MONETARIAS]` para utilizarlo después.
    df[COLUMNAS_MONETARIAS] = df[COLUMNAS_MONETARIAS].round(2)
    # Línea 0304: Calcula o asigna el contenido de `df[COLUMNAS_PORCENTAJE]` para utilizarlo después.
    df[COLUMNAS_PORCENTAJE] = df[COLUMNAS_PORCENTAJE].round(2)
    # Línea 0305: Devuelve el resultado calculado por la función.
    return df

## Validaciones y resumen

In [4]:
# Línea 0001: Define la función `validar_base` para encapsular esta parte del proceso.
def validar_base(df, n_esperado=N_FILAS):
    # Línea 0002: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert df.shape == (n_esperado, len(COLUMNAS)), df.shape
    # Línea 0003: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert list(df.columns) == COLUMNAS
    # Línea 0004: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert df["ID"].is_unique
    # Línea 0005: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert df["ID"].notna().all()
    # Línea 0006: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert df.columns[-1] == "mora"
    # Línea 0007: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert df["ID"].str.fullmatch(r"\d{10}").all()
    # Línea 0008: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert df["ID"].map(_validar_cedula_ecuatoriana).all()
    # Línea 0009: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert df["mora"].notna().all()
    # Línea 0010: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert set(df["mora"].unique()) == set(ETIQUETAS_MORA.values())
    # Línea 0011: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert all(
        # Línea 0012: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        df[columna].isna().any()
        # Línea 0013: Recorre cada elemento requerido para aplicar la operación de forma repetida.
        for columna in COLUMNAS
        # Línea 0014: Evalúa la condición y ejecuta el bloque únicamente cuando es verdadera.
        if columna not in COLUMNAS_COMPLETAS
    # Línea 0015: Cierra la estructura, llamada o colección iniciada previamente.
    )
    # Línea 0016: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert all(df[columna].notna().all() for columna in COLUMNAS_COMPLETAS)
    # Línea 0017: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert df["ID"].isna().sum() == 0
    # Línea 0018: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert df["mora"].isna().sum() == 0
    # Línea 0019: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert set(df["vivienda"].dropna().unique()) == {"PROPIA", "ARRENDADA"}
    # Línea 0020: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert set(df["zona"].dropna().unique()) == {
        # Línea 0021: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        "QUITO", "GUAYAQUIL", "AMBATO", "IBARRA", "CUENCA", "SANTO DOMINGO"
    # Línea 0022: Cierra la estructura, llamada o colección iniciada previamente.
    }
    # Línea 0023: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert (df["variacion_depositos_ingresos"].dropna() > 0).all()

    # Línea 0024: Calcula o asigna el contenido de `intervalos` para utilizarlo después.
    intervalos = {
        # Línea 0025: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        0: (0, 0), 1: (1, 7), 2: (8, 30),
        # Línea 0026: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        3: (31, 60), 4: (61, 90), 5: (91, 240),
    # Línea 0027: Cierra la estructura, llamada o colección iniciada previamente.
    }
    # Línea 0028: Calcula o asigna el contenido de `filas_atraso` para utilizarlo después.
    filas_atraso = df.dropna(subset=["atraso"]).copy()
    # Línea 0029: Calcula o asigna el contenido de `filas_atraso["nivel_mora"]` para utilizarlo después.
    filas_atraso["nivel_mora"] = filas_atraso["mora"].map(NIVEL_MORA_POR_TEXTO)
    # Línea 0030: Recorre cada elemento requerido para aplicar la operación de forma repetida.
    for clase, (minimo, maximo) in intervalos.items():
        # Línea 0031: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        valores = filas_atraso.loc[filas_atraso["nivel_mora"] == clase, "atraso"]
        # Línea 0032: Comprueba esta condición y detiene la ejecución si no se cumple.
        assert valores.between(minimo, maximo).all()

    # Línea 0033: Calcula o asigna el contenido de `filas_eventos` para utilizarlo después.
    filas_eventos = df.dropna(subset=["atrasos_6m_12m", "atrasos_12m"])
    # Línea 0034: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert (filas_eventos["atrasos_6m_12m"] <= filas_eventos["atrasos_12m"]).all()


# Línea 0035: Define la función `imprimir_resumen` para encapsular esta parte del proceso.
def imprimir_resumen(df, ruta_salida):
    # Línea 0036: Muestra este resultado para facilitar la revisión del usuario.
    print("\nBASE SINTÉTICA GENERADA CORRECTAMENTE")
    # Línea 0037: Muestra este resultado para facilitar la revisión del usuario.
    print(f"Archivo: {ruta_salida}")
    # Línea 0038: Muestra este resultado para facilitar la revisión del usuario.
    print(f"Dimensiones: {df.shape[0]:,} filas x {df.shape[1]} columnas")
    # Línea 0039: Muestra este resultado para facilitar la revisión del usuario.
    print("\nDistribución de mora:")
    # Línea 0040: Calcula o asigna el contenido de `orden_mora` para utilizarlo después.
    orden_mora = list(ETIQUETAS_MORA.values())
    # Línea 0041: Calcula o asigna el contenido de `resumen_mora` para utilizarlo después.
    resumen_mora = pd.DataFrame(
        # Línea 0042: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        {
            # Línea 0043: Registra el valor correspondiente a `n` dentro de la configuración.
            "n": df["mora"].value_counts().reindex(orden_mora),
            # Línea 0044: Registra el valor correspondiente a `porcentaje` dentro de la configuración.
            "porcentaje": (
                # Línea 0045: Calcula o asigna el contenido de `100 * df["mora"].value_counts(normalize` para utilizarlo después.
                100 * df["mora"].value_counts(normalize=True).reindex(orden_mora)
            # Línea 0046: Cierra la estructura, llamada o colección iniciada previamente.
            ).round(2),
        # Línea 0047: Cierra la estructura, llamada o colección iniciada previamente.
        }
    # Línea 0048: Cierra la estructura, llamada o colección iniciada previamente.
    )
    # Línea 0049: Muestra este resultado para facilitar la revisión del usuario.
    print(resumen_mora.to_string())
    # Línea 0050: Muestra este resultado para facilitar la revisión del usuario.
    print("\nValores faltantes por variable:")
    # Línea 0051: Calcula o asigna el contenido de `resumen_na` para utilizarlo después.
    resumen_na = pd.DataFrame(
        # Línea 0052: Ejecuta esta instrucción como parte del cálculo o validación descritos.
        {
            # Línea 0053: Registra el valor correspondiente a `faltantes` dentro de la configuración.
            "faltantes": df.isna().sum(),
            # Línea 0054: Registra el valor correspondiente a `porcentaje` dentro de la configuración.
            "porcentaje": (100 * df.isna().mean()).round(2),
        # Línea 0055: Cierra la estructura, llamada o colección iniciada previamente.
        }
    # Línea 0056: Cierra la estructura, llamada o colección iniciada previamente.
    )
    # Línea 0057: Muestra este resultado para facilitar la revisión del usuario.
    print(resumen_na.to_string())

## Generar y exportar el CSV

In [5]:
# Línea 0001: Calcula o asigna el contenido de `df` para utilizarlo después.
df = generar_base()
# Línea 0002: Ejecuta esta instrucción como parte del cálculo o validación descritos.
validar_base(df)

# Línea 0003: Calcula o asigna el contenido de `carpeta_salida` para utilizarlo después.
# Se fuerza la ruta a /content/ para asegurar que el archivo se guarde en una ubicación esperada por `files.download`.
carpeta_salida = Path('/content')
# Línea 0004: Calcula o asigna el contenido de `ruta_salida` para utilizarlo después.
ruta_salida = carpeta_salida / NOMBRE_SALIDA
# Línea 0005: Exporta el DataFrame a CSV sin índice y conservando correctamente las tildes.
df.to_csv(ruta_salida, index=False, encoding='utf-8-sig', na_rep='')
# Línea 0006: Ejecuta esta instrucción como parte del cálculo o validación descritos.
imprimir_resumen(df, ruta_salida)


BASE SINTÉTICA GENERADA CORRECTAMENTE
Archivo: /content/dataset_mora.csv
Dimensiones: 21,500 filas x 34 columnas

Distribución de mora:
                            n  porcentaje
mora                                     
AL DÍA                  15481        72.0
MORA DE 1 A 7 DÍAS       2150        10.0
MORA DE 8 A 30 DÍAS      1505         7.0
MORA DE 31 A 60 DÍAS      967         4.5
MORA DE 61 A 90 DÍAS      752         3.5
MORA DE MÁS DE 90 DÍAS    645         3.0

Valores faltantes por variable:
                                              faltantes  porcentaje
ID                                                    0        0.00
atraso                                              431        2.00
vivienda                                            879        4.09
edad                                                207        0.96
dias_lab                                           1317        6.13
exp_sf                                             1721        8.00
nivel_ahorro      

## Comprobaciones finales

In [6]:
df = generar_base()
# Línea 0001: Muestra este resultado para facilitar la revisión del usuario.
print('Columnas:', len(df.columns))
# Línea 0002: Muestra este resultado para facilitar la revisión del usuario.
print('Última columna:', df.columns[-1])
# Línea 0003: Muestra este resultado para facilitar la revisión del usuario.
print('IDs únicos:', df['ID'].nunique())
# Línea 0004: Muestra este resultado para facilitar la revisión del usuario.
print('Cédulas válidas:', int(df['ID'].map(_validar_cedula_ecuatoriana).sum()))
# Línea 0005: Muestra este resultado para facilitar la revisión del usuario.
print('Vivienda:', sorted(df['vivienda'].dropna().unique().tolist()))
# Línea 0006: Muestra este resultado para facilitar la revisión del usuario.
print('Zonas:', sorted(df['zona'].dropna().unique().tolist()))
# Línea 0007: Muestra este resultado para facilitar la revisión del usuario.
print('Variación mínima positiva:', df['variacion_depositos_ingresos'].min())
# Línea 0008: Muestra este resultado para facilitar la revisión del usuario.
print('Etiquetas de mora:', list(ETIQUETAS_MORA.values()))
# Línea 0009: Muestra este resultado para facilitar la revisión del usuario.
print('Faltantes en columnas completas:')
# Línea 0010: Muestra este resultado para facilitar la revisión del usuario.
print(df[['tipo_credito', 'nivel_ahorro', 'deuda_sf', 'zona', 'saldo_pendiente_credito', 'mora']].isna().sum().to_string())
# Línea 0011: Muestra este resultado para facilitar la revisión del usuario.
print(df.head(5).to_string(index=False))

Columnas: 34
Última columna: mora
IDs únicos: 21500
Cédulas válidas: 21500
Vivienda: ['ARRENDADA', 'PROPIA']
Zonas: ['AMBATO', 'CUENCA', 'GUAYAQUIL', 'IBARRA', 'QUITO', 'SANTO DOMINGO']
Variación mínima positiva: 0.85
Etiquetas de mora: ['AL DÍA', 'MORA DE 1 A 7 DÍAS', 'MORA DE 8 A 30 DÍAS', 'MORA DE 31 A 60 DÍAS', 'MORA DE 61 A 90 DÍAS', 'MORA DE MÁS DE 90 DÍAS']
Faltantes en columnas completas:
tipo_credito               0
nivel_ahorro               0
deuda_sf                   0
zona                       0
saldo_pendiente_credito    0
mora                       0
        ID  atraso  vivienda  edad  dias_lab  exp_sf nivel_ahorro  ingreso  linea_sf  deuda_sf  score      zona clasif_sbs    nivel_educ  mora_max_3m  mora_max_6m  mora_max_12m  atrasos_6m_12m  atrasos_12m      dti  cuota_ingreso  saldo_promedio_3m  variacion_depositos_ingresos  num_entidades_deuda  nuevos_creditos_3m_6m  nuevos_creditos_6m  porcentaje_utilizacion_linea  saldo_pendiente_credito  cuotas_restantes tipo_credi

## Descargar el archivo en Google Colab

In [7]:
# Línea 0001: Inicia un bloque protegido para controlar errores del entorno.
try:
    # Línea 0002: Importa la utilidad de Google Colab que permite descargar archivos.
    from google.colab import files
    # Línea 0003: Calcula o asigna el contenido de `carpeta_salida` para utilizarlo después.
    carpeta_salida = Path('/content') if Path('/content').exists() else Path.cwd()
    # Línea 0004: Calcula o asigna el contenido de `ruta_salida` para utilizarlo después.
    ruta_salida = carpeta_salida / NOMBRE_SALIDA
    # Línea 0005: Solicita al navegador la descarga del CSV recién generado.
    files.download(str(ruta_salida))
# Línea 0006: Controla el caso en que la ejecución no se realiza dentro de Google Colab.
except ImportError:
    # Línea 0007: Muestra este resultado para facilitar la revisión del usuario.
    print('Fuera de Colab: el CSV quedó guardado en', ruta_salida)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>